In [ ]:
!pip install mpld3

In [ ]:
from matplotlib import pyplot as pl
%matplotlib inline
import mpld3
mpld3.enable_notebook()
import numpy as np

# Demo of the standard degridder

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/processing_functions_tutorials/imaging/demo_standard_degrid.ipynb)

This notebook degrids (predicts) **model visibilities** from a **model sky image**.

Workflow:
1. Load the NGC 5921 processing set (it also carries a `VISIBILITY_MODEL` predicted by CASA and the model image `mod_image.npy`).
2. Build an image dataset (`img_xds`) with a `model` data group holding a `SKY_MODEL` image.
3. Fourier transform the model sky image into a model UV grid with `fft_norm_img_xds`.
4. Degrid that UV grid onto the measurement set's `(u, v)` coordinates with `get_visibility_grid_single_field`, writing `VISIBILITY_MODEL` into the MS.
5. Compare the AstroVIPER-predicted `VISIBILITY_MODEL` with the CASA prediction (amplitude vs. time).

## API update

This tutorial has been modernized to the current processing-function API. The old
`make_standard_degrid_grid`, `degrid_spheroid_ms4`, and
`astroviper.core.imaging.fft.fft_lm_to_uv` helpers have been removed. Degridding is
now driven by `get_visibility_grid_single_field`, with the model UV grid built by
`fft_norm_img_xds` and a prolate-spheroidal kernel from
`create_prolate_spheroidal_kernel_1D`.

## Assumptions

The centre pixel is assumed to be the phase centre.

---
## API

## Install AstroVIPER



In [ ]:
from importlib.metadata import version
import os

try:
    os.system("pip install --upgrade astroviper")

    import astroviper

    print("Using astroviper version", version("astroviper"))

except ImportError as exc:
    print(f"Could not import astroviper: {exc}")

In [ ]:
from astroviper.processing_functions.imaging.get_visibility_grid import (
    get_visibility_grid_single_field,
)
from astroviper.processing_functions.imaging.fft_normalize_prolate_spheriodal_gridder import (
    fft_norm_img_xds,
)
from astroviper.processing_functions.imaging.gridding_convolution_functions.gcf_prolate_spheroidal import (
    create_prolate_spheroidal_kernel_1D,
)

help(get_visibility_grid_single_field)

## Degrid a model image to model visibilities

We download the NGC 5921 processing set (which already contains a CASA-predicted
`VISIBILITY_MODEL` and the model image `mod_image.npy` used to make it), build a
model sky image, transform it to a model UV grid, degrid it onto the measurement
set's baselines, and compare the result with CASA.

### Download the data

Install `gdown` and download the NGC 5921 processing set (which carries the CASA
model prediction) together with the model image `mod_image.npy`.

In [ ]:
!pip install gdown
import gdown
gdown.download(id='1ybyBA6e5XiGXja8CI9yMBEFaMcLz82gd', output='mod_image.npy')
gdown.download(id='19br3EYwdtu82iF4JkRaX-9u2_bhNAMjJ', output='lala.zip')
!unzip -o lala.zip

Load the processing set that carries the CASA model visibilities (used later for comparison).

In [ ]:
from xradio.measurement_set import load_processing_set
ngc_xdt = load_processing_set('ngc5921_casa_model.ps.zarr')
ngc_mod = ngc_xdt['ngc5921_model_0']

Load a second copy of the processing set; we predict the AstroVIPER model into this one.

In [ ]:
ngc_xdt2 = load_processing_set('ngc5921_casa_model.ps.zarr')
ngc_ms4 = ngc_xdt2['ngc5921_model_0']

### The model image

Load `mod_image.npy` (the model CASA predicted from) and take a look at it.

In [ ]:
mod_arr = np.load('mod_image.npy')
pl.imshow(mod_arr, vmax=0.001599)

In [ ]:
import xarray as xr
from astropy import units as u
from xradio.image import make_empty_sky_image

# Model image cell size is 15 arcsec; the RA (l) increment is negative by convention.
incr = (15 * u.arcsec).to('rad').value
image_size = list(mod_arr.shape[-2:])
cell_size = [-incr, incr]

# Phase centre, frequency and polarization axes are taken from the processing set,
# so the model image matches the measurement set's spectral/polarization layout.
combined = ngc_xdt.xr_ps.get_combined_field_and_source_xds()
phase_center = combined.FIELD_PHASE_CENTER_DIRECTION.sel(
    field_name=combined.attrs["center_field_name"]
).values
frequency_coords = ngc_xdt.xr_ps.get_freq_axis().values
pol_coords = list(ngc_mod.polarization.values)

img_xds = make_empty_sky_image(
    phase_center=phase_center,
    image_size=image_size,
    cell_size=cell_size,
    frequency_coords=frequency_coords,
    pol_coords=pol_coords,
    time_coords=[0],
)
img_xds.attrs["type"] = "image_dataset"

# Broadcast the (l, m) model image across time, frequency and polarization.
# The source is unpolarized, so the Stokes-I model goes into every correlation plane.
sky = np.broadcast_to(
    mod_arr,
    (1, len(frequency_coords), len(pol_coords), image_size[0], image_size[1]),
).astype(np.float64).copy()
img_xds["SKY_MODEL"] = xr.DataArray(
    sky, dims=("time", "frequency", "polarization", "l", "m")
)

# Register a "model" data group whose "sky" role points to SKY_MODEL.
img_xds = img_xds.xr_img.add_data_group(
    new_data_group_name="model",
    new_data_group={"sky": "SKY_MODEL", "description": "degrid demo", "date": "2026"},
)
img_xds

### Form the model UV grid

Build the 1-D prolate-spheroidal gridding kernel and Fourier transform the model
sky image (`SKY_MODEL`) into a model UV grid (`VISIBILITY_MODEL`), stored back in
`img_xds`. `fft_norm_img_xds` applies the gridding-correction and zero-padding
(`fft_padding`) expected by the degridder.

In [ ]:
cgk_1D = create_prolate_spheroidal_kernel_1D(100, 7)

image_params = {"image_size": image_size, "fft_padding": 1.2}
img_xds = fft_norm_img_xds(
    img_xds,
    image_params=image_params,
    image_data_group_in_name="model",
    image_data_group_out_name="model",
    image_data_group_out_modified={"visibility": "VISIBILITY_MODEL"},
    image_data_variables_keep=["sky"],
    data_variables_to_process=["sky"],
    num_threads=1,
    fft_backend="scipy",
    complex_dtype=np.complex128,
)
img_xds

### Degrid onto the measurement set

`get_visibility_grid_single_field` samples the model UV grid at each visibility's
`(u, v)` coordinate with the prolate-spheroidal kernel and writes the predicted
`VISIBILITY_MODEL` into the measurement set. We first drop the CASA prediction from
this copy so the function allocates a fresh (zero-initialised) output array.

In [ ]:
import time

# Start from a clean model-visibility array (drop the CASA prediction in this copy).
if "VISIBILITY_MODEL" in ngc_ms4.data_vars:
    del ngc_ms4["VISIBILITY_MODEL"]

t0 = time.time()
get_visibility_grid_single_field(
    ngc_ms4,
    cgk_1D,
    img_xds,
    ms_data_group_in_name="base",
    ms_data_group_out_name="model",
    ms_data_group_out_modified={"correlated_data": "VISIBILITY_MODEL"},
    image_data_group_in_name="model",
    overwrite=True,
    chan_mode="cube",
    fft_padding=1.2,
    num_threads=1,
)
print(f"degrid time: {time.time() - t0:.3f} s")

In [ ]:
this_mod = ngc_ms4.VISIBILITY_MODEL
casa_mod = ngc_mod.VISIBILITY_MODEL

# Pick a valid baseline / channel to compare amplitude vs. time.
bl = min(200, this_mod.sizes["baseline_id"] - 1)
fchan = this_mod.sizes["frequency"] // 2
sel = dict(baseline_id=bl, frequency=fchan, polarization=0)

pl.plot(this_mod.isel(**sel).time, np.abs(this_mod.isel(**sel)), '+r', label='viper')
pl.plot(casa_mod.isel(**sel).time, np.abs(casa_mod.isel(**sel)), 'og', label='casa')
pl.xlabel('time')
pl.ylabel('|VISIBILITY_MODEL|')
pl.title(f'baseline {bl}, channel {fchan}')
pl.legend()

**AstroVIPER vs CASA**

The AstroVIPER prediction (`get_visibility_grid_single_field`) tracks the CASA
prediction closely. Small differences remain, largely from how each code rounds
visibility `(u, v)` coordinates onto grid pixels; the Van Cittert–Zernike relation
is not exactly a 2-D FFT, so the recovered flux also deviates with distance from
the phase centre.